# IND320 Compulsory Assignment 4

Elias Vinsrygg

This notebook documents Compulsory Assignment 4 in the course IND320.

## 1. Project links

- Public GitHub repository: [IND320-EliasVinsrygg](https://github.com/EliasVinsrygg/IND320-EliasVinsrygg)
- Published Streamlit app: [IND320 Energy and Weather](https://ind320-eliasvinsrygg.streamlit.app/)
- CA4 screencast: *Link to be added.*

## 2. Work Log

*To be written during the project, covering notebook work, Streamlit development, experiments and findings, and the selected bonus task.*

## 3. AI Usage

Codex assisted with recording the CA4 requirements in AGENTS.md and creating the initial administrative structure of this notebook.

*This description will be updated to reflect the assistance actually used during CA4.*

## 4. NB data

In [4]:
# Reading the API's CSV response directly from memory.
from io import StringIO

import pandas as pd
import requests


def download_exchange_rates(start_date=None, end_date=None):
    """Return NB's daily USD/NOK and EUR/NOK data for the selected dates."""
    # Using today's date and one calendar year of history when dates are omitted.
    end = pd.Timestamp.today().normalize() if end_date is None else pd.Timestamp(end_date)
    start = end - pd.DateOffset(years=1) if start_date is None else pd.Timestamp(start_date)
    if pd.isna(start) or pd.isna(end) or start > end:
        raise ValueError("Dates must be valid, with start_date on or before end_date.")

    # Passing the selected dates as query parameters to the daily exchange-rate endpoint.
    url = "https://data.norges-bank.no/api/data/EXR/B.USD+EUR.NOK.SP"
    params = {
        "startPeriod": start.strftime("%Y-%m-%d"),
        "endPeriod": end.strftime("%Y-%m-%d"),
        "format": "csv",
        "locale": "en",
    }
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    # Parsing the semicolon-separated CSV while retaining the API's column names.
    return pd.read_csv(StringIO(response.text), sep=";")


In [2]:
# Downloading the default one-year period and previewing the returned table.
exchange_rates = download_exchange_rates()
exchange_rates.head()


,FREQ,Frequency,BASE_CUR,Base Currency,QUOTE_CUR,Quote Currency,TENOR,Tenor,DECIMALS,CALCULATED,UNIT_MULT,Unit Multiplier,COLLECTION,Collection Indicator,TIME_PERIOD,OBS_VALUE
0,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-11-24,10.2170
1,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-11-25,10.2571
2,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-11-26,10.2330
3,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-11-27,10.1873
4,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-11-28,10.1716


## 5. Timezone investigation


| Source | Time reference | Handling |
| --- | --- | --- |
| [Magasinstatistikk](https://webfileservice.nve.no/API/PublishedFiles/Download/64edbe40-b965-4c90-a114-88162ffff454/201903383/3435075) | Weekly date; measurement Sunday 24:00 or nearest earlier available reading. The date field gives no UTC offset. | Keep the week label; do not interpret it as Sunday 00:00 UTC. Confirm the timezone if exact alignment is needed. |
| [ENTSO-E](https://transparencyplatform.zendesk.com/hc/en-us/articles/12786431986964-Response-Time-Zone) | API timestamps use UTC. The source says: "In response, time is always expressed in UTC." | Retain UTC |
| [Open-Meteo](https://open-meteo.com/en/docs/historical-weather-api) | Our stored responses use GMT, with UTC offset 0. Precipitation covers the preceding hour; temperature is instantaneous. | Retain UTC and account for interval meaning when aggregating. |
| [Norges Bank](https://www.norges-bank.no/en/news-events/news/Press-releases/2015/Press-release-07122015/) | Daily observation date; metadata state 14:15 CET. Publication is around 16:00 CET. | Keep the date; do not treat it as midnight UTC or as the publication time. |

Use UTC for actual timestamps and `Europe/Oslo` for Norwegian calendar boundaries: UTC+1 in winter and UTC+2 in summer. Agree common week boundaries before joining sources, and only use data available at the forecast origin. Existing data can be reused; no new downloads are needed here.

Course reference: [Time formats](../../Curriculum/D2Dbook/3_Data_quality/3_Preprocessing/3_Time_formats.ipynb), "ISO standard" and "Time conversion".


In [5]:
import pandas as pd

# Converting winter and summer UTC timestamps while preserving the same instants.
utc_times = pd.to_datetime(["2024-01-15 12:00", "2024-07-15 12:00"], utc=True)
timezone_example = pd.DataFrame({"UTC": utc_times, "Europe/Oslo": utc_times.tz_convert("Europe/Oslo")})
timezone_example


,UTC,Europe/Oslo
0,2024-01-15 12:00:00+00:00,2024-01-15 13:00:00+01:00
1,2024-07-15 12:00:00+00:00,2024-07-15 14:00:00+02:00


## 6. Independent data preparation, 2016-2025

Run the shared setup cell first, then the nine source-specific code cells below in order. Raw CA4 responses are saved separately in `data/ca4/2016_2025/`; reruns reuse these new checkpoints, not CA2/CA3 data or MongoDB. Downloads are not started until their cells run. The only reused inputs are the fixed 100 NO3 coordinates and the private ENTSO-E token.

All four outputs use Sunday dates for complete Monday-Sunday weeks in `Europe/Oslo`. Partial boundary weeks are excluded. Reservoir filling is the week-end observation (%); temperature and exchange rates are means; precipitation (mm) and flow energy (MWh) are sums. Missing flow coverage remains NaN. These are retrospective features: publication delays and unknown future inputs must be handled before forecasting.

Course: [Time formats: Time conversion](../../Curriculum/D2Dbook/3_Data_quality/3_Preprocessing/3_Time_formats.ipynb), [REST: Retrieving data (GET)](../../Curriculum/D2Dbook/2_Data_sources/3_APIs/2_REST.ipynb), [ML approach: Pandas' rolling() and shifts](../../Curriculum/D2Dbook/4_Machine_Learning/2_Forecasting/2_ML_approach.ipynb).

### Getting ready for data downloading

Defining shared imports, paths, complete target weeks and the download/checkpoint helper. Run this cell before the source-specific download and processing cells. No API requests or data files are created by this setup cell.


In [6]:
# Preparing shared dependencies for the four independent data sources.
from pathlib import Path
import json
import time
import tomllib
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import requests

# Locating shared configuration without changing the notebook's working folder.
PROJECT_ROOT = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "Compulsory Assignments/assets/weather_locations.json").is_file()
)
DATA_FOLDER = PROJECT_ROOT / "Compulsory Assignments/data/ca4/2016_2025"
START_DATE, END_DATE = pd.Timestamp("2016-01-01"), pd.Timestamp("2026-01-01")
LOCAL_ZONE = "Europe/Oslo"

# Keeping only full local calendar weeks, with Sunday as their date label.
weeks = pd.date_range(START_DATE, END_DATE, freq="W-SUN", inclusive="left", name="date")
weeks = weeks[(weeks - pd.Timedelta(days=6) >= START_DATE)
              & (weeks + pd.Timedelta(days=1) <= END_DATE)]
week_starts = (weeks - pd.Timedelta(days=6)).tz_localize(LOCAL_ZONE).tz_convert("UTC")
week_ends = (weeks + pd.Timedelta(days=1)).tz_localize(LOCAL_ZONE).tz_convert("UTC")
expected_week_hours = pd.Series((week_ends - week_starts) / pd.Timedelta(hours=1), index=weeks)


def download_text(url, params, filename):
    """Download one response, or reuse a CA4 checkpoint; never retry HTTP errors."""
    path = DATA_FOLDER / filename
    if path.is_file():
        return path.read_text(encoding="utf-8")
    # Hiding request URLs because ENTSO-E query parameters contain the API token.
    try:
        response = requests.get(url, params=params, timeout=(15, 180))
    except requests.RequestException:
        raise RuntimeError("API connection failed; completed CA4 files are preserved.") from None
    if response.status_code != 200:
        raise RuntimeError(f"HTTP {response.status_code}; no automatic retry. "
                           f"Retry-After: {response.headers.get('Retry-After', 'not supplied')}")
    response.encoding = "utf-8"
    # Saving completed responses atomically so an interruption leaves no partial checkpoint.
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(response.text, encoding="utf-8")
    temporary.replace(path)
    return response.text


### 6.1 Magasinstatistikk: download

[NVE API](https://api.nve.no/doc/magasinstatistikk/). Downloading the full reservoir history, or reusing the independent CA4 checkpoint.


In [7]:
# Downloading NVE's JSON directly; this endpoint returns the full reservoir history.
nve_raw = json.loads(download_text(
    "https://biapi.nve.no/magasinstatistikk/api/Magasinstatistikk/HentOffentligData",
    {}, "reservoirs.json",
))
print(f"NVE: {len(nve_raw)} source records.")


NVE: 14976 source records.


### 6.1 Magasinstatistikk: processing

Keeping NO3 identifiers, ISO calendar fields, filling, stored energy, capacity and weekly change for analysis. Filling ratios are converted to percent; changes are converted to percentage points. Energy and capacity remain in TWh. Retaining these columns does not select them as model inputs: same-week stored energy and change must not reveal the target filling during forecasting.


In [8]:
# Selecting NO3 (EL/3), retaining source fields for analysis and later feature selection.
reservoir_columns = {
    "dato_Id": "date",
    "omrType": "area_type",
    "omrnr": "area_number",
    "iso_aar": "iso_year",
    "iso_uke": "iso_week",
    "fyllingsgrad": "filling_percent",
    "fylling_TWh": "stored_energy_twh",
    "kapasitet_TWh": "capacity_twh",
    "fyllingsgrad_forrige_uke": "previous_week_filling_percent",
    "endring_fyllingsgrad": "weekly_change_pp",
}
reservoir = pd.DataFrame(nve_raw)
reservoir = reservoir.loc[
    (reservoir["omrType"] == "EL") & (reservoir["omrnr"] == 3),
    list(reservoir_columns),
].rename(columns=reservoir_columns).copy()

# Keeping weekly date labels without inventing UTC instants, and validating numeric fields.
reservoir["date"] = pd.to_datetime(reservoir["date"], errors="raise")
numeric_columns = [column for column in reservoir.columns if column not in {"date", "area_type"}]
reservoir[numeric_columns] = reservoir[numeric_columns].apply(pd.to_numeric, errors="raise")
if (reservoir["date"].isna().any() or reservoir["date"].duplicated().any()
        or not np.isfinite(reservoir[numeric_columns].to_numpy(dtype=float)).all()):
    raise ValueError("Reservoir dates must be unique and present, and numeric values must be finite.")

# Converting source ratios to percent and changes to percentage points; retaining TWh units.
percent_columns = ["filling_percent", "previous_week_filling_percent", "weekly_change_pp"]
reservoir[percent_columns] = reservoir[percent_columns] * 100
if (not reservoir[["filling_percent", "previous_week_filling_percent"]].apply(
        lambda column: column.between(0, 100)).all().all()
        or not reservoir["weekly_change_pp"].between(-100, 100).all()
        or not reservoir["capacity_twh"].gt(0).all()
        or not reservoir["stored_energy_twh"].ge(0).all()):
    raise ValueError("Unexpected filling percentages, weekly changes, stored energy or capacity.")

# Verifying the source ISO year/week against the date, including weeks around New Year.
iso_calendar = reservoir["date"].dt.isocalendar()
if (not reservoir["iso_year"].eq(iso_calendar["year"]).all()
        or not reservoir["iso_week"].eq(iso_calendar["week"]).all()
        or not reservoir["date"].dt.dayofweek.eq(6).all()):
    raise ValueError("Reservoir ISO fields must match their Sunday observation dates.")

# Checking every target week without interpolating gaps or discarding ISO week 53.
reservoir_weekly = reservoir.set_index("date").sort_index().reindex(weeks)
if reservoir_weekly.isna().any().any():
    raise ValueError("Missing NO3 reservoir weeks or fields; inspect the source before modelling.")
reservoir_weekly.head()


,area_type,area_number,iso_year,iso_week,filling_percent,stored_energy_twh,capacity_twh,previous_week_filling_percent,weekly_change_pp
date,,,,,,,,,
2016-01-10,EL,3,2016,1,68.995315,6.155025,8.920932,72.939220,-3.943902
2016-01-17,EL,3,2016,2,64.587426,5.761800,8.920932,68.995315,-4.407889
2016-01-24,EL,3,2016,3,59.888100,5.342576,8.920932,64.587426,-4.699326
2016-01-31,EL,3,2016,4,57.638526,5.141893,8.920932,59.888100,-2.249575
2016-02-07,EL,3,2016,5,54.853940,4.893482,8.920932,57.638526,-2.784586


### 6.2 Norges Bank: download

[NB currency API](https://data.norges-bank.no/api/data/EXR/B.USD+EUR.NOK.SP?startPeriod=2016-01-01&endPeriod=2025-12-31&format=csv&locale=en).

In [9]:
from io import StringIO

# Downloading daily USD/NOK and EUR/NOK for the entire independent CA4 period.
currency_raw = pd.read_csv(StringIO(download_text(
    "https://data.norges-bank.no/api/data/EXR/B.USD+EUR.NOK.SP",
    {"startPeriod": "2016-01-01", "endPeriod": "2025-12-31", "format": "csv", "locale": "en"},
    "currency.csv",
)), sep=";")
print(f"NB: {len(currency_raw)} source records.")


NB: 5032 source records.


### 6.2 Norges Bank: processing

In [10]:
# Keeping observation dates as dates; the API provides no per-row UTC timestamp.
currency = currency_raw[["TIME_PERIOD", "BASE_CUR", "QUOTE_CUR", "UNIT_MULT", "OBS_VALUE"]].copy()
currency["date"] = pd.to_datetime(currency["TIME_PERIOD"], errors="raise")
currency["rate"] = pd.to_numeric(currency["OBS_VALUE"], errors="raise")
if (not currency["QUOTE_CUR"].eq("NOK").all()
        or not pd.to_numeric(currency["UNIT_MULT"], errors="raise").eq(0).all()
        or set(currency["BASE_CUR"]) != {"USD", "EUR"}
        or not np.isfinite(currency["rate"]).all() or not currency["rate"].gt(0).all()
        or currency.duplicated(["date", "BASE_CUR"]).any()):
    raise ValueError("Unexpected currency units, values or duplicate observations.")

# Averaging available business-day observations, without inventing weekend/holiday rates.
currency_daily = currency.pivot(index="date", columns="BASE_CUR", values="rate").sort_index()
currency_weekly = currency_daily.resample("W-SUN").mean().reindex(weeks).rename(
    columns={"USD": "usd_nok", "EUR": "eur_nok"})
currency_weekly.columns.name = None
if currency_weekly.isna().any().any():
    raise ValueError("A target week has no exchange-rate observations.")
currency_weekly.head()


,eur_nok,usd_nok
date,,
2016-01-10,9.64730,8.91384
2016-01-17,9.61804,8.84878
2016-01-24,9.62910,8.85532
2016-01-31,9.46522,8.70568
2016-02-07,9.51122,8.62502


### 6.3 ENTSO-E: download

Downloading physical flows between NO3 and SE2, NO5, NO1 and NO4, with import and export defined relative to NO3. These are eight bidding-zone series, replacing the national country-pair inputs for CA4. [ENTSO-E area codes](https://transparencyplatform.zendesk.com/hc/en-us/articles/15885757676308-Area-List-with-Energy-Identification-Code-EIC) identify the requested zones; [API parameters](https://transparencyplatform.zendesk.com/hc/en-us/articles/15696716612372-Request-Parameters) define the queries.

Monthly requests limit response sizes: 960 requests for 2016-2025 before checkpoint reuse, with a short pause before new requests. Responses are stored under `entsoe/NO3/`, separately from country-pair files. Missing historical coverage is not assumed to be zero. HTTP errors stop the download without retries; completed files remain available for a later rerun.


In [16]:
# Reading the existing token privately; no Cassandra, Spark or MongoDB is involved.
with (PROJECT_ROOT / ".streamlit/secrets.toml").open("rb") as secrets_file:
    entsoe_token = tomllib.load(secrets_file)["entsoe"]["security_token"]

# Mapping the selected bidding zones to ENTSO-E EIC codes, rather than country codes.
no3_code = "10YNO-3--------J"
neighbour_codes = {
    "SE2": "10Y1001A1001A45N",
    "NO5": "10Y1001A1001A48H",
    "NO1": "10YNO-1--------2",
    "NO4": "10YNO-4--------9",
}
month_edges = pd.date_range(START_DATE, END_DATE, freq="MS", tz="UTC")
entsoe_files = []
for neighbour, neighbour_code in neighbour_codes.items():
    # Defining export as NO3 -> neighbour and import as neighbour -> NO3.
    for direction, origin, destination in (("export", no3_code, neighbour_code),
                                           ("import", neighbour_code, no3_code)):
        for start, end in zip(month_edges[:-1], month_edges[1:]):
            # Keeping these checkpoints separate from any national-flow downloads.
            filename = f"entsoe/NO3/{neighbour}_{direction}_{start:%Y_%m}.xml"
            if not (DATA_FOLDER / filename).is_file():
                time.sleep(0.3)
            download_text("https://web-api.tp.entsoe.eu/api",
                          {"securityToken": entsoe_token, "documentType": "A11",
                           "out_Domain": origin, "in_Domain": destination,
                           "periodStart": start.strftime("%Y%m%d%H%M"),
                           "periodEnd": end.strftime("%Y%m%d%H%M")}, filename)
            entsoe_files.append((filename, neighbour, direction, origin, destination, start, end))
        print(f"ENTSO-E: NO3 / {neighbour} {direction}, monthly responses saved.", flush=True)
del entsoe_token


ENTSO-E: NO3 / SE2 export, monthly responses saved.
ENTSO-E: NO3 / SE2 import, monthly responses saved.
ENTSO-E: NO3 / NO5 export, monthly responses saved.
ENTSO-E: NO3 / NO5 import, monthly responses saved.
ENTSO-E: NO3 / NO1 export, monthly responses saved.
ENTSO-E: NO3 / NO1 import, monthly responses saved.
ENTSO-E: NO3 / NO4 export, monthly responses saved.
ENTSO-E: NO3 / NO4 import, monthly responses saved.


### 6.3 ENTSO-E: processing

Following CA2's A01/A03 interval parsing: A01 values cover one resolution step, while A03 values last until the next change or the period end. [ENTSO-E curve types](https://transparencyplatform.zendesk.com/hc/en-us/articles/30262342482961-CurveType-A01-vs-CurveType-A03) and [UTC API timestamps](https://transparencyplatform.zendesk.com/hc/en-us/articles/12786431986964-Response-Time-Zone) describe the source format.

Converting MW to MWh using each interval's duration and splitting at local Monday boundaries in `Europe/Oslo`. Only fully covered weeks receive energy totals; missing or incomplete weeks remain NaN, while published zero flows remain zero. Import and export are retained separately for each neighbour: for example, `no3_se2_export_mwh` means NO3 -> SE2, and `no3_se2_import_mwh` means SE2 -> NO3. The eight columns share the target Sunday dates used by the other sources. Historical flow values are candidate features, not known future inputs.

Course: [Time formats](../../Curriculum/D2Dbook/3_Data_quality/3_Preprocessing/3_Time_formats.ipynb), "ISO standard" and "Time conversion"; [REST](../../Curriculum/D2Dbook/2_Data_sources/3_APIs/2_REST.ipynb), "Retrieving data (GET)".


In [17]:
def parse_flow_response(text, origin, destination, request_start, request_end):
    """Read published MW intervals, clipping them to one non-overlapping request."""
    root = ET.fromstring(text)
    if root.tag.endswith("Acknowledgement_MarketDocument"):
        reason = root.findtext(".//{*}Reason/{*}text") or "Unknown acknowledgement"
        if "no matching data" in reason.lower() or "no data" in reason.lower():
            return []
        raise ValueError(f"ENTSO-E rejected a query: {reason}")
    rows = []
    for series in root.findall(".//{*}TimeSeries"):
        if (series.findtext("{*}out_Domain.mRID") != origin
                or series.findtext("{*}in_Domain.mRID") != destination
                or series.findtext("{*}quantity_Measure_Unit.name") != "MAW"):
            raise ValueError("Unexpected ENTSO-E bidding zones, direction or power unit.")
        curve = series.findtext("{*}curveType")
        if curve not in {"A01", "A03"}:
            raise ValueError("Unsupported ENTSO-E curve type.")
        for period in series.findall("{*}Period"):
            begin = pd.to_datetime(period.findtext("{*}timeInterval/{*}start"), utc=True)
            finish = pd.to_datetime(period.findtext("{*}timeInterval/{*}end"), utc=True)
            step = pd.Timedelta(period.findtext("{*}resolution"))
            if step not in (pd.Timedelta(minutes=15), pd.Timedelta(minutes=30), pd.Timedelta(hours=1)):
                raise ValueError("Unexpected ENTSO-E resolution.")
            points = period.findall("{*}Point")
            positions = [int(point.findtext("{*}position")) for point in points]
            if positions != sorted(set(positions)):
                raise ValueError("ENTSO-E positions must be ordered and unique.")
            # Interpreting A01 as fixed steps and A03 as values lasting until the next change.
            for i, point in enumerate(points):
                start = begin + (positions[i] - 1) * step
                end = (begin + (positions[i + 1] - 1) * step
                       if curve == "A03" and i + 1 < len(points)
                       else finish if curve == "A03" else start + step)
                power = float(point.findtext("{*}quantity"))
                if not begin <= start < end <= finish or not np.isfinite(power):
                    raise ValueError("Invalid ENTSO-E interval or power.")
                start, end = max(start, request_start), min(end, request_end)
                if start < end:
                    rows.append((start, end, power))
    return rows


def weekly_flow_energy(blocks):
    """Split non-overlapping UTC power intervals at local week boundaries."""
    blocks = blocks.sort_values("start")
    if blocks["start"].lt(blocks["end"].shift()).any():
        raise ValueError("Overlapping ENTSO-E blocks; do not sum duplicate publications.")
    pieces = []
    for start, end, power in blocks[["start", "end", "power"]].itertuples(index=False, name=None):
        # Splitting blocks crossing Monday midnight, accounting for daylight saving time.
        while start < end:
            label = start.tz_convert(LOCAL_ZONE).tz_localize(None).to_period("W-SUN").end_time.normalize()
            boundary = (label + pd.Timedelta(days=1)).tz_localize(LOCAL_ZONE).tz_convert("UTC")
            stop = min(end, boundary)
            hours = (stop - start) / pd.Timedelta(hours=1)
            pieces.append((label, power * hours, hours))
            start = stop
    if not pieces:
        return pd.Series(np.nan, index=weeks)
    totals = pd.DataFrame(pieces, columns=["date", "energy", "hours"]).groupby("date").sum().reindex(weeks)
    # Leaving incomplete weeks missing, including weeks around historical connection gaps.
    complete = np.isclose(totals["hours"], expected_week_hours)
    return totals["energy"].where(complete)


# Reading only NO3 neighbour files; empty responses remain explicitly documented.
flow_rows, missing_flow_queries = [], []
for filename, neighbour, direction, origin, destination, start, end in entsoe_files:
    rows = parse_flow_response((DATA_FOLDER / filename).read_text(encoding="utf-8"),
                               origin, destination, start, end)
    if not rows:
        missing_flow_queries.append((neighbour, direction, start))
    flow_rows.extend((neighbour, direction, *row) for row in rows)
flows = pd.DataFrame(flow_rows, columns=["neighbour", "direction", "start", "end", "power"])
if flows.empty:
    raise ValueError("No ENTSO-E physical-flow observations were returned.")
entsoe_weekly = pd.DataFrame(index=weeks)
for neighbour in neighbour_codes:
    for direction in ("import", "export"):
        selected = flows.loc[flows["neighbour"].eq(neighbour) & flows["direction"].eq(direction)]
        column = f"no3_{neighbour.lower()}_{direction}_mwh"
        entsoe_weekly[column] = weekly_flow_energy(selected)
print(f"ENTSO-E: {len(missing_flow_queries)} NO3 neighbour/direction/month queries without data.")
print("Missing weeks per feature:")
print(entsoe_weekly.isna().sum())
entsoe_weekly.head()


ENTSO-E: 2 NO3 neighbour/direction/month queries without data.
Missing weeks per feature:
no3_se2_import_mwh     0
no3_se2_export_mwh     0
no3_no5_import_mwh    21
no3_no5_export_mwh    21
no3_no1_import_mwh    16
no3_no1_export_mwh    16
no3_no4_import_mwh    16
no3_no4_export_mwh    16
dtype: int64


,no3_se2_import_mwh,no3_se2_export_mwh,no3_no5_import_mwh,no3_no5_export_mwh,no3_no1_import_mwh,no3_no1_export_mwh,no3_no4_import_mwh,no3_no4_export_mwh
date,,,,,,,,
2016-01-10,21789.89,5820.7,NaN,NaN,23403.97,35.74,121260.39,0.0
2016-01-17,8443.55,13821.4,NaN,NaN,14591.25,2612.79,138625.10,0.0
2016-01-24,140.23,22375.0,NaN,NaN,4199.07,6546.81,126269.12,0.0
2016-01-31,73941.14,2850.3,NaN,NaN,10628.95,7470.27,109352.74,0.0
2016-02-07,69057.72,1468.4,NaN,NaN,17404.87,4095.16,113919.02,0.0


### 6.4 Open-Meteo: round one, 2016-2020

[Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api), [quotas](https://open-meteo.com/en/pricing). Fixed 100 NO3 points, ERA5, temperature and precipitation, GMT/UTC. New annual requests start at least 60 seconds apart; HTTP errors stop without retries. Five requests are about 2,610 weighted calls using the earlier calculator estimate (about 521-523 per year, not below 500). Quotas also include other use of the API.

Run round one and round two in order in the same notebook kernel. Round two automatically waits **65 minutes** before starting its downloads, so the cells can be queued in sequence. The wait runs again every time the round-two cell is executed. Annual CA4 files allow recovery after an interruption.

In [ ]:
# Reading only the fixed coordinates, independently of earlier weather datasets and app functions.
locations = np.asarray(json.loads((PROJECT_ROOT / "Compulsory Assignments/assets/weather_locations.json")
                                 .read_text(encoding="utf-8"))["regions"]["NO3"], dtype=float)
if locations.shape != (100, 2) or not np.isfinite(locations).all() or (np.abs(locations) > [180, 90]).any():
    raise ValueError("Expected 100 valid longitude/latitude pairs for NO3.")


# Recording the exact requested sample so later rounds cannot mix different coordinates.
location_file = DATA_FOLDER / "weather/NO3_locations.json"
if location_file.is_file():
    if json.loads(location_file.read_text(encoding="utf-8")) != locations.tolist():
        raise ValueError("The fixed NO3 sample differs from the sample saved for CA4.")
else:
    location_file.parent.mkdir(parents=True, exist_ok=True)
    location_file.write_text(json.dumps(locations.tolist()), encoding="utf-8")


def weather_hourly_mean(payload, year):
    """Validate 100 full-year responses and return equal-weight regional hourly means."""
    hours = pd.date_range(f"{year}-01-01", f"{year + 1}-01-01", freq="h", inclusive="left", tz="UTC")
    if not isinstance(payload, list) or len(payload) != 100:
        raise ValueError("Expected exactly 100 weather responses.")
    values = []
    for point in payload:
        frame = pd.DataFrame(point["hourly"])[["time", "temperature_2m", "precipitation"]]
        frame["time"] = pd.to_datetime(frame["time"], utc=True)
        if (not pd.DatetimeIndex(frame["time"]).equals(hours)
                or point.get("utc_offset_seconds") != 0
                or point["hourly_units"].get("temperature_2m") != "\u00b0C"
                or point["hourly_units"].get("precipitation") != "mm"):
            raise ValueError(f"Unexpected weather hours, timezone or units in {year}.")
        numeric = frame[["temperature_2m", "precipitation"]].to_numpy(dtype=float)
        if not np.isfinite(numeric).all() or (numeric[:, 1] < 0).any():
            raise ValueError(f"Missing/invalid weather values in {year}.")
        values.append(numeric)
    # Retaining identical grid series as separate sampled points with equal weights.
    return pd.DataFrame(np.mean(values, axis=0), index=hours,
                        columns=["temperature_c", "precipitation_mm"]).rename_axis("time")


def download_weather_years(years):
    """Save five annual CA4 weather responses, with 60-second request-start spacing."""
    files, last_start = [], None
    for year in years:
        filename = f"weather/NO3_{year}.json"
        path = DATA_FOLDER / filename
        if path.is_file():
            weather_hourly_mean(json.loads(path.read_text(encoding="utf-8")), year)
        else:
            if last_start is not None:
                time.sleep(max(0, 60 - (time.monotonic() - last_start)))
            print(f"Open-Meteo: requesting NO3/{year}.", flush=True)
            last_start = time.monotonic()
            payload = json.loads(download_text("https://archive-api.open-meteo.com/v1/archive",
                {"latitude": ",".join(map(str, locations[:, 1])),
                 "longitude": ",".join(map(str, locations[:, 0])),
                 "start_date": f"{year}-01-01", "end_date": f"{year}-12-31",
                 "hourly": "temperature_2m,precipitation", "models": "era5", "timezone": "GMT",
                 "temperature_unit": "celsius", "precipitation_unit": "mm"}, filename))
            try:
                weather_hourly_mean(payload, year)
            except (ValueError, KeyError, TypeError):
                # Removing only this invalid CA4 response so a later run can download it again.
                path.unlink()
                raise
        files.append(path)
        print(f"Open-Meteo: NO3/{year} validated and saved.", flush=True)
    return files


# Starting the first five-year round; running this cell makes live requests for missing files.
weather_round_one = download_weather_years(range(2016, 2021))
print("Round one finished. Round two starts with an automatic 65-minute wait.")


### 6.4 Open-Meteo: round two, 2021-2025 (automatic 65-minute wait)


In [ ]:
# Waiting before round two so both download cells can be queued in the same kernel.
print("Waiting 65 minutes before starting round two. Keep the notebook kernel running.", flush=True)
for minute in range(65):
    time.sleep(60)

# Starting the remaining five years after the automatic pause.
print("65-minute wait finished. Starting round two.", flush=True)
weather_round_two = download_weather_years(range(2021, 2026))
print("Round two finished. Both rounds can now be processed together.")


### 6.4 Open-Meteo: concatenate and process both rounds

In [ ]:
# Concatenating all ten validated annual regional series, regardless of notebook restarts.
weather_files = [DATA_FOLDER / f"weather/NO3_{year}.json" for year in range(2016, 2026)]
if not all(path.is_file() for path in weather_files):
    raise ValueError("Complete both download rounds before processing all ten years.")
weather_hourly = pd.concat([
    weather_hourly_mean(json.loads(path.read_text(encoding="utf-8")), year)
    for year, path in zip(range(2016, 2026), weather_files)
]).sort_index()
expected_hours = pd.date_range(START_DATE, END_DATE, freq="h", inclusive="left", tz="UTC")
if not weather_hourly.index.equals(expected_hours):
    raise ValueError("Weather years have gaps, overlaps or unexpected time coverage.")

# Matching local weeks: temperature is instantaneous; precipitation ends at its timestamp.
local_times = weather_hourly.index.tz_convert(LOCAL_ZONE)
temperature_labels = local_times.tz_localize(None).to_period("W-SUN").end_time.normalize()
rain_labels = (local_times - pd.Timedelta(1, unit="ns")).tz_localize(None).to_period("W-SUN").end_time.normalize()
temperature = weather_hourly["temperature_c"].groupby(temperature_labels).agg(["mean", "count"])
precipitation = weather_hourly["precipitation_mm"].groupby(rain_labels).agg(["sum", "count"])
if (not temperature["count"].reindex(weeks).eq(expected_week_hours).all()
        or not precipitation["count"].reindex(weeks).eq(expected_week_hours).all()):
    raise ValueError("Incomplete weather weeks; daylight-saving weeks may contain 167 or 169 hours.")
weather_weekly = pd.DataFrame({"temperature_c": temperature["mean"],
                               "precipitation_mm": precipitation["sum"]}).reindex(weeks)
print(f"Weather: {len(weather_hourly)} UTC hours; {len(weather_weekly)} complete local weeks.")
weather_weekly.head()
